# YOLO11 street issue training on Google Colab

Classes: `pothole`, `garbage`, `overflowing_bin`. This notebook never generates labels or downloads a dataset. Use your validated, prepared dataset, its attribution files, and trusted `yolo11n.pt` weights.

1. Choose **Runtime → Change runtime type → GPU**.
2. In Google Drive create `MyDrive/street-issues/` and upload `colab-training-code.zip`, `dataset.zip`, and `yolo11n.pt` as described in the training README.
3. Run setup and preflight cells. Inspect warnings and coverage.
4. Training is off by default. Only change `START_TRAINING` to `True` when you intend to start the long run. Keep resume off for a new run.

Drive output preserves completed-epoch checkpoints across Colab disconnects; it cannot recover an unsaved first epoch.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, zipfile, stat, json

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/street-issues')
CODE_ROOT = Path('/content/street_training_code')
DATA_ROOT = Path('/content/street_dataset')
WEIGHTS = BASE / 'yolo11n.pt'
RUN_NAME = 'street-issues-yolo11n-colab-01'  # New name for each fresh run.
OUTPUT = BASE / 'runs' / RUN_NAME
for filename in ('colab-training-code.zip', 'dataset.zip', 'yolo11n.pt'):
    assert (BASE / filename).is_file(), f'Upload {filename} to {BASE} first'

def extract_new(archive_path, destination):
    if destination.exists():
        raise FileExistsError(f'{destination} exists. Skip extraction if already done, or use a fresh runtime/path.')
    with zipfile.ZipFile(archive_path) as archive:
        for item in archive.infolist():
            target = (destination / item.filename).resolve()
            if not target.is_relative_to(destination.resolve()) or stat.S_ISLNK(item.external_attr >> 16):
                raise ValueError(f'Unsafe ZIP member: {item.filename}')
        archive.extractall(destination)

extract_new(BASE / 'colab-training-code.zip', CODE_ROOT)
extract_new(BASE / 'dataset.zip', DATA_ROOT)
assert (CODE_ROOT / 'ai_service/scripts/train.py').is_file(), 'Use the provided package_colab command'
assert (DATA_ROOT / 'data.yaml').is_file(), 'ZIP must contain data.yaml, images/ and labels/ directly, without an outer prepared/ folder'
os.chdir(CODE_ROOT)
print('Code:', CODE_ROOT, '\nData:', DATA_ROOT, '\nOutput:', OUTPUT)

In [ ]:
# Installs Python packages only; does not download any dataset or model weights.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(CODE_ROOT / 'ai_service/requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)

In [ ]:
import yaml
import torch

# Prepared Windows YAML may contain an absolute Windows path; relocate only the
# dataset root. Keep labels, original splits, attribution and manifests intact.
DATA_YAML = DATA_ROOT / 'data.yaml'
config = yaml.safe_load(DATA_YAML.read_text())
config['path'] = str(DATA_ROOT)
DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False))
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU fallback will be used. Select a GPU runtime before a long run if available.')

COMMON = [sys.executable, '-m', 'ai_service.scripts.train', '--data', str(DATA_YAML)]
# Append --allow-empty-labels ONLY if your dataset includes verified negatives.
# COMMON += ['--allow-empty-labels']
subprocess.run(COMMON + ['--weights', str(WEIGHTS), '--check', '--device', 'auto'], check=True)

## Explicitly start a new training run

The defaults are 50 total epochs, image size 640, seed 42 and batch 8 on CUDA. CPU batch is capped at 2. Edit values below to fit GPU memory; batch 4 or 2 is a reasonable retry if 8 runs out of memory. Preflight runs again before training. Test images are checked for integrity/leakage but are not used for validation or model selection.

The notebook does not assess whether your dataset is large or representative enough. Review class coverage and samples first. Avoid **Run all** after enabling training; execute the intended training cell yourself.

In [ ]:
START_TRAINING = False  # Change to True and run this cell only when ready.
EPOCHS, BATCH, IMAGE_SIZE, SEED = 50, 8, 640, 42
if START_TRAINING:
    subprocess.run(COMMON + [
        '--train', '--weights', str(WEIGHTS), '--output', str(OUTPUT),
        '--epochs', str(EPOCHS), '--batch', str(BATCH), '--imgsz', str(IMAGE_SIZE),
        '--seed', str(SEED), '--device', 'auto',
    ], check=True)
else:
    print('Training is disabled. No training has started.')

## Resume an interrupted run (optional)

After reconnecting, rerun setup/install/relocation/preflight cells with the **same dataset and RUN_NAME**. Skip the new-run training cell. Keep the whole Drive run directory, including `run.json` and `weights/last.pt`. The pipeline verifies unchanged image/label/split fingerprints and restores the original total epoch target, image size and seed. Resume starts from the last saved epoch; optimizer/scheduler state is restored by Ultralytics. A finalized, optimizer-stripped checkpoint cannot resume.

In [ ]:
RESUME_TRAINING = False  # Enable only when you intend to resume.
if RESUME_TRAINING:
    checkpoint = OUTPUT / 'weights/last.pt'
    assert checkpoint.is_file(), f'No completed-epoch checkpoint: {checkpoint}'
    subprocess.run(COMMON + ['--train', '--resume', str(checkpoint), '--device', 'auto', '--batch', str(BATCH)], check=True)
else:
    print('Resume is disabled.')

## Download the trained model and review metrics

`weights/best.pt` is selected using the validation split. Review `results.csv`, `results.png`, validation plots, `validation_metrics.json`, `training.log`, `console.log` and `status.json` in your Drive run directory. `best.pt`/`last.pt` only exist after real training saves them; this notebook never fabricates weights or metrics. Download the full run directory as well if you need logs and provenance.

In [ ]:
DOWNLOAD_BEST = False
if DOWNLOAD_BEST:
    from google.colab import files
    best = OUTPUT / 'weights/best.pt'
    assert best.is_file(), 'No best.pt yet. Training must save a checkpoint first.'
    files.download(str(best))
else:
    print('Download disabled. Set DOWNLOAD_BEST=True after training, or download best.pt directly from Drive.')